# EEG Workload Classification
Theta/Alpha ratio -> SVM classifier (low vs high memory load)

## 1. Install dependencies

In [7]:
%pip install mne mne-bids openneuro-py scikit-learn pymatreader -q

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Users\\saran\\AppData\\Local\\Programs\\Python\\Python310\\Lib\\site-packages\\~umpy\\.libs\\libopenblas.FB5AE2TYXYH2IJRDKGDGQ3XBKLKTF43H.gfortran-win_amd64.dll'
Consider using the `--user` option or check the permissions.


[notice] A new release of pip is available: 23.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Config

In [4]:
DATASET = "ds003838"
ROOT = "./eeg_data"
N_SUBJECTS = 5
FRONTAL_CHANNELS = ['Fz', 'F3', 'F4']
PARIETAL_CHANNELS = ['Pz', 'P3', 'P4']
THETA_BAND = (4, 8)
ALPHA_BAND = (8, 13)
FEATURES_FILE = "features.npz"

## 3. Data Manager

In [5]:
import os
import pandas as pd
import openneuro
from mne_bids import find_matching_paths


class DataManager:
    def __init__(self, dataset=DATASET, root=ROOT):
        self.dataset = dataset
        self.root = root
        os.makedirs(self.root, exist_ok=True)

    def ensure_downloaded(self, include):
        missing = []
        for f in include:
            path = os.path.join(self.root, f)
            if f.endswith("/eeg"):
                set_files = (
                    [x for x in os.listdir(path) if x.endswith(".set")]
                    if os.path.exists(path) else []
                )
                if not set_files:
                    missing.append(f)
            else:
                if not os.path.exists(path):
                    missing.append(f)

        if missing:
            print(f"Downloading missing: {missing}")
            openneuro.download(dataset=self.dataset, target_dir=self.root, include=missing)
        else:
            print("All required files already present, skipping download.")

    def get_valid_subject_ids(self, n_subjects=N_SUBJECTS):
        self.ensure_downloaded(["dataset_description.json", "participants.tsv"])
        participants = pd.read_csv(f"{self.root}/participants.tsv", sep="\t")
        valid = participants[participants['EEG_excluded'] == 'no']
        subject_ids = valid['participant_id'].str.replace('sub-', '').tolist()[:n_subjects]
        print("Using subjects:", subject_ids)
        return subject_ids

    def download_subjects_eeg(self, subject_ids):
        includes = [f"sub-{s}/eeg" for s in subject_ids]
        self.ensure_downloaded(includes)

    def get_bids_paths(self, task="memory"):
        paths = find_matching_paths(
            self.root, datatypes="eeg", suffixes="eeg",
            tasks=task, extensions=".set"
        )
        print(paths)
        return paths

    def prepare(self, n_subjects=N_SUBJECTS, task="memory"):
        subject_ids = self.get_valid_subject_ids(n_subjects)
        self.download_subjects_eeg(subject_ids)
        return self.get_bids_paths(task)

ModuleNotFoundError: No module named 'openneuro'

## 4. Feature Extractor

In [ ]:
import numpy as np
import mne
from mne_bids import read_raw_bids
from mne.time_frequency import psd_array_welch
import gc

class FeatureExtractor:
    def __init__(self,
                 frontal_channels=FRONTAL_CHANNELS,
                 parietal_channels=PARIETAL_CHANNELS,
                 theta_band=THETA_BAND,
                 alpha_band=ALPHA_BAND):
        self.frontal_channels = frontal_channels
        self.parietal_channels = parietal_channels
        self.theta_band = theta_band
        self.alpha_band = alpha_band

    @staticmethod
    def _band_power(data, sfreq, band):
        psds, _ = psd_array_welch(data, sfreq, fmin=band[0], fmax=band[1])
        return psds.mean(-1)

    def _load_and_preprocess(self, bids_path):
        raw = read_raw_bids(bids_path, verbose=False)
        raw.load_data()
        raw.pick(self.frontal_channels + self.parietal_channels)
        raw.filter(1., 40.)
        raw.set_eeg_reference('average')
        return raw

    def _epoch(self, raw):
        events, event_id = mne.events_from_annotations(raw)
        low_keys = [k for k in event_id if 'memory' in k and 'in 5 digit sequence' in k]
        high_keys = [k for k in event_id if 'memory' in k and 'in 13 digit sequence' in k]
        epochs = mne.Epochs(raw, events, event_id, tmin=-0.2, tmax=1.0,
                            baseline=(None, 0), preload=True)
        return epochs, low_keys, high_keys

    def _channel_indices(self, raw):
        frontal = [ch for ch in self.frontal_channels if ch in raw.ch_names]
        parietal = [ch for ch in self.parietal_channels if ch in raw.ch_names]
        f_idx = [raw.ch_names.index(c) for c in frontal]
        p_idx = [raw.ch_names.index(c) for c in parietal]
        return f_idx, p_idx

    def extract_ratio(self, epochs, raw, f_idx, p_idx):
        sfreq = raw.info['sfreq']
        theta = self._band_power(epochs.get_data(), sfreq, self.theta_band)[:, f_idx].mean(1)
        alpha = self._band_power(epochs.get_data(), sfreq, self.alpha_band)[:, p_idx].mean(1)
        return theta / alpha

    def process_subject(self, bids_path):
        raw = self._load_and_preprocess(bids_path)
        epochs, low_keys, high_keys = self._epoch(raw)
        f_idx, p_idx = self._channel_indices(raw)
        low = epochs[low_keys]
        high = epochs[high_keys]
        ratio_low = self.extract_ratio(low, raw, f_idx, p_idx)
        ratio_high = self.extract_ratio(high, raw, f_idx, p_idx)
        return ratio_low, ratio_high

    # def process_all(self, bids_paths):
    #     all_ratio_low, all_ratio_high, subject_groups_low, subject_groups_high = [], [], [], []
    #     for i, bp in enumerate(bids_paths):
    #         ratio_low, ratio_high = self.process_subject(bp)
    #         all_ratio_low.append(ratio_low)
    #         all_ratio_high.append(ratio_high)
    #         subject_groups_low.append(np.full(len(ratio_low), i))
    #         subject_groups_high.append(np.full(len(ratio_high), i))
    #         gc.collect()   # ADD THIS LINE
    #     return (np.concatenate(all_ratio_low), np.concatenate(all_ratio_high),
    #             np.concatenate(subject_groups_low), np.concatenate(subject_groups_high))
    def process_all(self, bids_paths, save_dir="."):
      for i, bp in enumerate(bids_paths):
          ratio_low, ratio_high = self.process_subject(bp)
          np.savez(f"{save_dir}/features_subj{i}.npz",
                  ratio_low=ratio_low, ratio_high=ratio_high,
                  groups_low=np.full(len(ratio_low), i),
                  groups_high=np.full(len(ratio_high), i))
          del ratio_low, ratio_high
          gc.collect()
      print(f"Saved {len(bids_paths)} subject feature files.")

    def save_features(self, ratio_low, ratio_high, groups_low, groups_high, path=FEATURES_FILE):
        np.savez(path, ratio_low=ratio_low, ratio_high=ratio_high,
                groups_low=groups_low, groups_high=groups_high)

## 5. Run extraction (downloads data + computes features)

In [ ]:
# dm = DataManager()
# bids_paths = dm.prepare()

# fe = FeatureExtractor()
# ratio_low, ratio_high, groups_low, groups_high = fe.process_all(bids_paths)
# fe.save_features(ratio_low, ratio_high, groups_low, groups_high)
dm = DataManager()
bids_paths = dm.prepare()

fe = FeatureExtractor()
fe.process_all(bids_paths)

## 6. Classifier (fixed: scaling + class_weight + multiple metrics)

In [ ]:
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold, cross_val_score


class WorkloadClassifier:
    def __init__(self, kernel='rbf', cv=5):
        self.clf = Pipeline([
            ('scaler', StandardScaler()),
            ('svc', SVC(kernel=kernel, class_weight='balanced'))
        ])
        self.cv = cv

    @staticmethod
    def load_features(path=FEATURES_FILE):
        data = np.load(path)
        return data['ratio_low'], data['ratio_high'], data['groups_low'], data['groups_high']

    @staticmethod
    def build_dataset(ratio_low, ratio_high, groups_low, groups_high):
        X = np.concatenate([ratio_low, ratio_high]).reshape(-1, 1)
        y = np.array([0]*len(ratio_low) + [1]*len(ratio_high))
        groups = np.concatenate([groups_low, groups_high])
        return X, y, groups

    def evaluate(self, X, y, groups):
        gkf = GroupKFold(n_splits=self.cv)
        acc = cross_val_score(self.clf, X, y, cv=gkf, groups=groups, scoring='accuracy')
        bal_acc = cross_val_score(self.clf, X, y, cv=gkf, groups=groups, scoring='balanced_accuracy')
        f1 = cross_val_score(self.clf, X, y, cv=gkf, groups=groups, scoring='f1')

        print(f"Accuracy:          {acc.mean():.2f} ± {acc.std():.2f}")
        print(f"Balanced Accuracy: {bal_acc.mean():.2f} ± {bal_acc.std():.2f}")
        print(f"F1-score:          {f1.mean():.2f} ± {f1.std():.2f}")
        print(f"Class distribution: {np.bincount(y)}")
        return {'accuracy': acc, 'balanced_accuracy': bal_acc, 'f1': f1}

## 7. Run classification

In [ ]:
clf = WorkloadClassifier()
ratio_low, ratio_high, groups_low, groups_high = clf.load_features()
X, y, groups = clf.build_dataset(ratio_low, ratio_high, groups_low, groups_high)
results = clf.evaluate(X, y, groups)